# MathLive input widgets showcase

This notebook explains the **semantic MathLive layer**. The goal is to show how the toolkit turns user-facing math input into canonical identifiers and SymPy expressions.

What you should learn here:

- what an `ExpressionContext` registers
- how `IdentifierInput` and `ExpressionInput` use that context
- why MathJSON is the preferred transport format
- which errors are intentional safeguards rather than missing features

## Setup

The notebook only adds the repository `src/` folder to `sys.path`.

In [5]:
import import_setup

In [6]:
import sympy as sp
from IPython.display import display

from gu_toolkit import NamedFunction
from gu_toolkit.identifiers import symbol
from gu_toolkit.mathlive import (
    ExpressionContext,
    ExpressionInput,
    IdentifierInput,
    MathJSONParseError,
    mathjson_to_identifier,
)

## 1. Build an explicit semantic context

`ExpressionContext` lists which names are symbols, which are functions, and how they should render. It is the contract between the backend and the widgets. 

The backend stores **canonical names** such as `velocity`, `theta__x`, and `Force_t`. Display LaTeX is derived from those names instead of being stored as the identity itself.

In [11]:
# Add markdown explanation that we are first going to use symbol
help(symbol)
#BUG: the docstring of symbol is useless. "Public semantic-math helper callable for symbol" does not tell me anything. '
# Also, does sympy provide a symbol export? If so, this should be noted and explicitly stated that this should override and remain API compatible
# Provide link to sympy doc in the documentation. 

#BUG: stop using "This API accepts the parameters declared in its Python signature" This is a useless sentence. All Parameter descriptions must be meaningful and informative
# This applies to other fields of the documentation. For example: how am I supposed to know what latex_expr does? WTF am I supposed to do with kwargs>? 

Help on function symbol in module gu_toolkit.identifiers.policy:

symbol(name: 'str', *, latex_expr: 'str | None' = None, **kwargs: 'Any') -> 'sp.Symbol'
    Public semantic-math helper callable for symbol.

    Full API
    --------
    ``symbol(...)``

    Parameters
    ----------
    This API accepts the parameters declared in its Python signature.

    Returns
    -------
    object
        Result produced by this API.

    Optional arguments
    ------------------
    Optional arguments follow the defaults declared in the Python signature when present.

    Architecture note
    -----------------
    This API lives in ``gu_toolkit.identifiers.policy`` and participates in the toolkit's canonical identifier, parsing, or semantic math-input infrastructure.

    Examples
    --------
    Basic use::

        result = symbol(...)

    Learn more / explore
    --------------------
    - Start with ``docs/guides/api-discovery.md`` for a task-oriented map of the package.
    - Example no

In [7]:
#BUG: This is very bad. There is no explanation for the code! WTF does manifest do? WTF is it such a strange object: a dict of dicts. Shouldn't it be .latex instead of ['latex']
# Revise this 

#BUG: change the way explanatory cells are organized. Provide meaningful printed representation FIRST 

#BUG: why do we need to say functions= in the call for ExpressionContext. Can't that information be deduced from the type of the sympy object?

@NamedFunction
def Force(x):
    return x

velocity = symbol('velocity')
theta_x_literal = symbol('theta__x')
a_12 = symbol('a_1_2')
x = symbol('x')

ctx = ExpressionContext.from_symbols(
    [velocity, theta_x_literal, a_12, x],
    functions=[Force, 'Force_t', 'Force__x'],
    include_named_functions=False,
)

#BUG WTF is a transport_manifest? Come on! Your focus is on explaining and teaching this toolkit. You are doing a really bad job with this code. Rethink your approach. Improve!

manifest = ctx.transport_manifest(field_role='expression')
symbols_by_name = {entry['name']: entry for entry in manifest['symbols']}
functions_by_name = {entry['name']: entry for entry in manifest['functions']}

assert symbols_by_name['velocity']['latex'] == r'\mathrm{velocity}'
assert symbols_by_name['theta__x']['latex'] == r'\mathrm{theta\_x}'
assert functions_by_name['Force']['latexHead'] == r'\operatorname{Force}'
assert functions_by_name['Force_t']['template'] == r'\operatorname{Force}_{t}(#0)'

summary = {
    'fieldRole': manifest['fieldRole'],
    'symbols': [
        {key: symbols_by_name[name][key] for key in ('name', 'latex', 'atoms')}
        for name in ('velocity', 'theta__x', 'a_1_2')
    ],
    'functions': [
        {key: functions_by_name[name][key] for key in ('name', 'latexHead', 'template')}
        for name in ('Force', 'Force_t', 'Force__x')
    ],
}
summary

{'fieldRole': 'expression',
 'symbols': [{'name': 'velocity',
   'latex': '\\mathrm{velocity}',
   'atoms': ['velocity']},
  {'name': 'theta__x', 'latex': '\\mathrm{theta\\_x}', 'atoms': ['theta_x']},
  {'name': 'a_1_2', 'latex': 'a_{1,2}', 'atoms': ['a', '1', '2']}],
 'functions': [{'name': 'Force',
   'latexHead': '\\operatorname{Force}',
   'template': '\\operatorname{Force}(#0)'},
  {'name': 'Force_t',
   'latexHead': '\\operatorname{Force}_{t}',
   'template': '\\operatorname{Force}_{t}(#0)'},
  {'name': 'Force__x',
   'latexHead': '\\operatorname{Force\\_x}',
   'template': '\\operatorname{Force\\_x}(#0)'}]}

## 2. `IdentifierInput` normalizes what the user typed

The widget can receive either plain LaTeX text or MathJSON. In both cases, the backend should reduce the result to the same canonical identifier.

This cell also doubles as an integration check: the asserts make sure the documented examples keep working over time.

In [ ]:
identifier_widget = IdentifierInput(context=ctx, value=r'a_{1,2}')
display(identifier_widget)

assert identifier_widget.parse_value() == 'a_1_2'

identifier_widget.value = ''
identifier_widget.math_json = ['Subscript', 'a', ['Tuple', 1, 2]]
assert identifier_widget.parse_value() == 'a_1_2'

'IdentifierInput normalizes both LaTeX and MathJSON to the same canonical name.'

## 3. `ExpressionInput` keeps known names atomic

Without a context, names like `velocity` could be parsed as a product of single-letter symbols. With a context, the expression widget keeps registered names atomic and keeps registered functions callable.

In [ ]:
expression_widget = ExpressionInput(
    context=ctx,
    value=r'\operatorname{Force}(\mathrm{theta\_x}) + 2\mathrm{velocity}',
)
display(expression_widget)

parsed_from_text = expression_widget.parse_value()
assert parsed_from_text == Force(theta_x_literal) + 2 * velocity
assert r'\operatorname{Force}' in ctx.render_latex(parsed_from_text)
assert r'\mathrm{theta\_x}' in ctx.render_latex(parsed_from_text)

parsed_from_text

## 4. MathJSON is the preferred transport

When the MathLive Compute Engine is available, the widget can send a structured MathJSON payload instead of relying only on string parsing. This avoids many classes of ambiguity and gives the backend a cleaner, more testable boundary.

In [ ]:
expression_widget.value = ''
expression_widget.math_json = [
    'Add',
    ['Force', 'theta__x'],
    ['Multiply', 'velocity', 2],
]

parsed_from_mathjson = expression_widget.parse_value()
assert parsed_from_mathjson == Force(theta_x_literal) + 2 * velocity
assert parsed_from_mathjson == parsed_from_text

parsed_from_mathjson

## 5. Some errors are intentional safeguards

A transport spelling such as `theta_x` is ambiguous when no context says whether it should mean a real subscript or a literal underscore inside the atom. In that situation the toolkit **raises** instead of guessing.

This is an example of the general design philosophy: fail early at the semantic boundary, not later inside unrelated plotting code.

In [ ]:
try:
    mathjson_to_identifier('theta_x')
except MathJSONParseError as exc:
    ambiguous_message = str(exc)
else:
    raise AssertionError('Expected an ambiguity error for an unregistered single-underscore name.')

assert 'ambiguous' in ambiguous_message.lower()
assert mathjson_to_identifier('theta__x', context=ctx) == 'theta__x'

ambiguous_message

## Try your own variations

Good experiments after reading this notebook:

- register a new symbol such as `sigma_1` or `sigma__x` and compare the rendered LaTeX
- add a new function name such as `Energy_t` and inspect the transport manifest again
- change the widget text to plain canonical input (`Force(theta__x) + 2*velocity`) and compare it with the explicit LaTeX form
- compare what happens with `theta_x` before and after registering it in the context